Моделирование мотивов в PWM. Сканирование хромосомы человека

In [1]:
pip install numpy matplotlib scipy biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.0 MB/s eta 0:00:00


Загрузка последовательности хромосомы человека

In [11]:
import urllib.request
import gzip
import shutil
from Bio import SeqIO

filename_gz = "chr1.fa.gz"
filename_fa = "chr1.fa"
url_hg38 = "http://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr1.fa.gz"


urllib.request.urlretrieve(url_hg38, filename_gz)

with gzip.open(filename_gz, 'rb') as f_in:
    with open(filename_fa, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)


records = list(SeqIO.parse(filename_fa, "fasta"))
chr1_record = records[0]
print(f"Загружена запись: ID={chr1_record.id}, длина={len(chr1_record.seq)}")


sequence = str(chr1_record.seq[:1000000])
print(f"Длина извлеченной последовательности: {len(sequence)}")

Загружена запись: ID=chr1, длина=248956422
Длина извлеченной последовательности: 1000000


In [12]:
from Bio import motifs
from Bio import SeqIO
import numpy as np
import urllib.request
import gzip
import shutil
import os

# Создаём мотив
sites = ["GAGGTAAAC", "TCCGTAAGC", "CAGGTTGGA", "ACAGTCAGC",
         "TAGGTCAGC", "CAGGTCAGC", "CAGGTCGAT", "CAGGTCAGC",
         "CAGGTCAGC", "CAGGTTGGC"]

motif = motifs.create(sites)
motif.pseudocounts = 0.1
motif.background = {'A': 0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}

pwm_bio = motif.pwm
print("PWM (Biopython):\n", pwm_bio)

records = SeqIO.parse(filename, "fasta")
chr1_record = next(records)
sequence = str(chr1_record.seq[:1000000])
print(f"Длина последовательности: {len(sequence)}")

# -Поиск сайтов (прямая + обратная цепь)
threshold = 5.0

# PSSM для прямой цепи
pssm = motif.pssm

# Функция для безопасного извлечения (pos, score) из результата search
def extract_hits(search_result):
    """Извлекает список (position, score) из результатов search."""
    result = []
    for hit in search_result:
        if isinstance(hit, tuple):
            pos, score = hit
        else:
            pos = hit.position
            score = hit.score
        result.append((pos, score))
    return result

# Ищем на прямой цепи
hits_forward = extract_hits(pssm.search(sequence, threshold=threshold))

# Ищем на обратной цепи
pssm_rc = pssm.reverse_complement()
hits_reverse = extract_hits(pssm_rc.search(sequence, threshold=threshold))

# Формируем общий список (позиция, цепь, скор)
all_hits = []
for pos, score in hits_forward:
    all_hits.append((pos, '+', score))
for pos, score in hits_reverse:
    all_hits.append((pos, '-', score))

all_hits.sort(key=lambda x: x[0])

print(f"\nНайдено хитов (всего): {len(all_hits)}")
print("Первые 10 хитов (позиция, цепь, скор):")
for pos, strand, score in all_hits[:10]:
    print(f"  {pos:8d}  {strand}  {score:.4f}")

# Сохраняем в файл
with open("hits_threshold5.txt", "w") as f:
    f.write("Pos\tStrand\tScore\n")
    for pos, strand, score in all_hits:
        f.write(f"{pos}\t{strand}\t{score:.4f}\n")

print("\nРезультаты сохранены в 'hits_threshold5.txt'.")

PWM (Biopython):
         0      1      2      3      4      5      6      7      8
A:   0.11   0.78   0.11   0.01   0.01   0.20   0.68   0.20   0.11
C:   0.59   0.20   0.11   0.01   0.01   0.59   0.01   0.01   0.78
G:   0.11   0.01   0.78   0.97   0.01   0.01   0.30   0.78   0.01
T:   0.20   0.01   0.01   0.01   0.97   0.20   0.01   0.01   0.11

Длина последовательности: 1000000

Найдено хитов (всего): 7754
Первые 10 хитов (позиция, цепь, скор):
   -989500  +  11.9790
   -988998  +  8.3477
   -988679  +  5.4967
   -988596  +  8.7834
   -988132  +  7.4429
   -988124  -  8.6104
   -988109  +  7.0351
   -987988  +  6.1022
   -987971  +  9.2514
   -987776  -  5.0876

Результаты сохранены в 'hits_threshold5.txt'.


In [13]:

from google.colab import files
files.download('hits_threshold5.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>